In [1]:

# Day 9 - Step 1: Project Risk Data Readiness Audit

from pathlib import Path
import pandas as pd

project_dir = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

# Inspect existing prospect, quotation, and revenue datasets.
files_to_check = [
    ("data/raw/NexziByte_Final_Scored.csv", "Raw prospect data"),
    ("data/clean/Day3_Prospects_Clean.csv", "Clean prospect data"),
    ("data/synthetic/Day7_Synthetic_Quotation_Dataset.csv",
     "Synthetic quotation data"),
    ("data/synthetic/Day8_Synthetic_Monthly_Revenue.csv",
     "Synthetic revenue data"),
]

project_keywords = [
    "project", "milestone", "start_date", "target_end_date",
    "actual_end_date", "planned_effort", "actual_effort",
    "change_request", "dependency", "defect", "resource",
    "allocation", "utilization", "delivery_cost", "margin",
    "revenue", "client_delay", "delay_days", "status"
]

audit_rows = []

for relative_path, dataset_type in files_to_check:
    file_path = project_dir / relative_path

    if not file_path.exists():
        audit_rows.append({
            "File": relative_path,
            "Dataset_Type": dataset_type,
            "File_Exists": False,
            "Rows": None,
            "Columns": None,
            "Project_Related_Columns": "",
            "Data_Status": "File not found"
        })
        continue

    df = pd.read_csv(file_path)
    matched_columns = [
        col for col in df.columns
        if any(keyword in col.lower() for keyword in project_keywords)
    ]

    audit_rows.append({
        "File": relative_path,
        "Dataset_Type": dataset_type,
        "File_Exists": True,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Project_Related_Columns": ", ".join(matched_columns),
        "Data_Status": (
            "Synthetic demonstration only"
            if "synthetic" in relative_path.lower()
            else "Existing dataset; business validity not yet confirmed"
        )
    })

audit_df = pd.DataFrame(audit_rows)

output_path = (
    project_dir / "data" / "processed"
    / "Day9_Project_Risk_Data_Readiness_Audit.csv"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_df.to_csv(output_path, index=False)

print("DAY 9 - PROJECT RISK DATA READINESS AUDIT")
print(audit_df.to_string(index=False))
print("\nAudit saved to:")
print(output_path)


DAY 9 - PROJECT RISK DATA READINESS AUDIT
                                               File             Dataset_Type  File_Exists  Rows  Columns Project_Related_Columns                                           Data_Status
                data/raw/NexziByte_Final_Scored.csv        Raw prospect data         True   180       25     Data Quality Status Existing dataset; business validity not yet confirmed
                data/clean/Day3_Prospects_Clean.csv      Clean prospect data         True   180       28     Data Quality Status Existing dataset; business validity not yet confirmed
data/synthetic/Day7_Synthetic_Quotation_Dataset.csv Synthetic quotation data         True   300       12            Quote_Status                          Synthetic demonstration only
  data/synthetic/Day8_Synthetic_Monthly_Revenue.csv   Synthetic revenue data         True    36        4                 Revenue                          Synthetic demonstration only

Audit saved to:
D:\Nexzibyte Solution\nexz

In [2]:

# Day 9 - Step 2: Define Targets and Required Project Fields

import pandas as pd

# Target definitions are provisional modelling specifications.
# Confirm with the business owner before using real outcomes.

target_definitions = pd.DataFrame([
    {
        "Target": "Project_Delayed",
        "Definition": (
            "1 if Actual_End_Date exceeds Target_End_Date "
            "by the approved tolerance; otherwise 0"
        ),
        "Required_Fields": "Target_End_Date, Actual_End_Date",
        "Current_Status": "Unavailable in audited datasets",
        "Approval_Status": "Business confirmation pending"
    },
    {
        "Target": "Effort_Overrun",
        "Definition": (
            "1 if Actual_Effort exceeds Planned_Effort "
            "by the approved tolerance; otherwise 0"
        ),
        "Required_Fields": "Planned_Effort, Actual_Effort",
        "Current_Status": "Unavailable in audited datasets",
        "Approval_Status": "Business confirmation pending"
    },
    {
        "Target": "Low_Margin_Risk",
        "Definition": (
            "1 if forecast gross margin falls below "
            "the management-approved threshold"
        ),
        "Required_Fields": (
            "Project_Revenue, Forecast_Delivery_Cost, "
            "Approved_Margin_Threshold"
        ),
        "Current_Status": "Project-level values unavailable",
        "Approval_Status": "Finance confirmation pending"
    }
])

required_fields = pd.DataFrame([
    ("Project_ID", "Project identifier", "Project"),
    ("Target_End_Date", "Planned completion date", "Schedule"),
    ("Actual_End_Date", "Actual completion date", "Schedule"),
    ("Planned_Effort", "Budgeted delivery hours", "Effort"),
    ("Actual_Effort", "Consumed delivery hours", "Effort"),
    ("Milestones_Planned", "Milestones scheduled by the cutoff", "Delivery"),
    ("Milestones_Completed", "Milestones completed by the cutoff", "Delivery"),
    ("Change_Request_Count", "Approved or logged scope changes", "Scope"),
    ("Dependency_Delay_Days", "Delay caused by dependencies", "Dependencies"),
    ("Client_Delay_Days", "Delay attributable to client dependencies", "Client"),
    ("Resource_Utilization", "Resource utilization at the cutoff", "Resources"),
    ("Project_Revenue", "Approved project revenue", "Finance"),
    ("Forecast_Delivery_Cost", "Estimated total delivery cost", "Finance"),
    ("Actual_Delivery_Cost", "Recorded delivery cost", "Finance"),
    ("Data_Cutoff_Date", "Date when prediction features were observed", "Governance")
], columns=["Field", "Definition", "Category"])

targets_path = (
    project_dir / "data" / "processed"
    / "Day9_Project_Risk_Target_Definitions.csv"
)
fields_path = (
    project_dir / "data" / "processed"
    / "Day9_Required_Project_Risk_Fields.csv"
)

target_definitions.to_csv(targets_path, index=False)
required_fields.to_csv(fields_path, index=False)

print("TARGET DEFINITIONS")
print(target_definitions.to_string(index=False))

print("\nREQUIRED PROJECT FIELDS")
print(required_fields.to_string(index=False))

print("\nSaved target definitions to:", targets_path)
print("Saved required fields to:", fields_path)


TARGET DEFINITIONS
         Target                                                                          Definition                                                    Required_Fields                   Current_Status               Approval_Status
Project_Delayed 1 if Actual_End_Date exceeds Target_End_Date by the approved tolerance; otherwise 0                                   Target_End_Date, Actual_End_Date  Unavailable in audited datasets Business confirmation pending
 Effort_Overrun    1 if Actual_Effort exceeds Planned_Effort by the approved tolerance; otherwise 0                                      Planned_Effort, Actual_Effort  Unavailable in audited datasets Business confirmation pending
Low_Margin_Risk            1 if forecast gross margin falls below the management-approved threshold Project_Revenue, Forecast_Delivery_Cost, Approved_Margin_Threshold Project-level values unavailable  Finance confirmation pending

REQUIRED PROJECT FIELDS
                 Field              

In [3]:

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# DAY 9 - STEP 3: SYNTHETIC PROJECT-RISK DATASET
# ============================================================

# 1. Project directory
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

# 2. Reproducible synthetic data
rng = np.random.default_rng(42)
n = 300

# 3. Generate project start dates
start_dates = pd.date_range(
    start="2023-01-01",
    end="2025-12-01",
    periods=n
).normalize()

# 4. Project planning information
planned_days = rng.integers(30, 181, n)
planned_effort = rng.integers(200, 2001, n).astype(float)

# Convert dates to DatetimeIndex
target_end_dates = (
    start_dates
    + pd.to_timedelta(planned_days, unit="D")
)

milestones_planned = rng.integers(4, 21, n)

# 5. Project factors observable during execution
change_requests = rng.poisson(2, n)
dependency_delay = rng.integers(0, 21, n)
client_delay = rng.integers(0, 16, n)
resource_utilization = rng.uniform(0.55, 1.05, n)

milestones_completed = np.array([
    rng.integers(0, milestone_count + 1)
    for milestone_count in milestones_planned
])

# 6. Generate actual effort outcomes
effort_multiplier = (
    0.85
    + 0.025 * change_requests
    + 0.008 * dependency_delay
    + 0.010 * client_delay
    + np.maximum(resource_utilization - 0.80, 0) * 0.8
    + rng.normal(0, 0.12, n)
)

actual_effort = (
    planned_effort
    * np.maximum(effort_multiplier, 0.60)
)

# 7. Generate actual completion dates
extra_delay = (
    0.8 * dependency_delay
    + 0.9 * client_delay
    + 2.0 * change_requests
    + np.maximum(resource_utilization - 0.85, 0) * 35
    + rng.normal(0, 8, n)
)

actual_end_dates = (
    target_end_dates
    + pd.to_timedelta(
        np.maximum(
            np.round(extra_delay), -10
        ).astype(int),
        unit="D"
    )
)

# FIX: Convert both date indexes to NumPy datetime arrays.
# Subtract them, then extract the integer day values.
delay_days = (
    actual_end_dates.to_numpy(dtype="datetime64[ns]")
    - target_end_dates.to_numpy(dtype="datetime64[ns]")
) / np.timedelta64(1, "D")

delay_days = delay_days.astype(int)

# 8. Generate illustrative financial values
project_revenue = rng.uniform(150000, 1500000, n)

forecast_cost_ratio = rng.uniform(0.55, 0.90, n)

forecast_delivery_cost = (
    project_revenue * forecast_cost_ratio
)

actual_cost_multiplier = (
    actual_effort / planned_effort
) * rng.uniform(0.95, 1.10, n)

actual_delivery_cost = (
    forecast_delivery_cost * actual_cost_multiplier
)

# 9. Calculate provisional target labels
# Illustrative assumptions only; business/finance approval pending.
# Delay: more than 7 days late.
# Effort overrun: more than 10% above planned effort.
# Low margin: forecast gross margin below 20%.

effort_overrun_pct = (
    actual_effort - planned_effort
) / planned_effort

forecast_gross_margin = (
    project_revenue - forecast_delivery_cost
) / project_revenue

project_delayed = (
    delay_days > 7
).astype(int)

effort_overrun = (
    effort_overrun_pct > 0.10
).astype(int)

low_margin_risk = (
    forecast_gross_margin < 0.20
).astype(int)

# 10. Define feature observation cutoff dates
data_cutoff_dates = (
    start_dates
    + pd.to_timedelta(
        (planned_days * 0.60).astype(int),
        unit="D"
    )
)

# 11. Build the synthetic dataset
project_data = pd.DataFrame({
    "Project_ID": [
        f"PRJ-{i:04d}"
        for i in range(1, n + 1)
    ],
    "Project_Start_Date": start_dates,
    "Target_End_Date": target_end_dates,
    "Actual_End_Date": actual_end_dates,
    "Planned_Effort": planned_effort.round(2),
    "Actual_Effort": actual_effort.round(2),
    "Milestones_Planned": milestones_planned,
    "Milestones_Completed": milestones_completed,
    "Change_Request_Count": change_requests,
    "Dependency_Delay_Days": dependency_delay,
    "Client_Delay_Days": client_delay,
    "Resource_Utilization": resource_utilization.round(3),
    "Project_Revenue": project_revenue.round(2),
    "Forecast_Delivery_Cost": forecast_delivery_cost.round(2),
    "Actual_Delivery_Cost": actual_delivery_cost.round(2),
    "Data_Cutoff_Date": data_cutoff_dates,
    "Delay_Days": delay_days,
    "Effort_Overrun_Pct": effort_overrun_pct.round(4),
    "Forecast_Gross_Margin": forecast_gross_margin.round(4),
    "Project_Delayed": project_delayed,
    "Effort_Overrun": effort_overrun,
    "Low_Margin_Risk": low_margin_risk,
    "Synthetic_Data_Flag": True,
    "Data_Use": "Synthetic demonstration only"
})

# 12. Save the dataset
output_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

project_data.to_csv(
    output_path,
    index=False
)

# 13. Validate the dataset
print("=" * 60)
print("DAY 9 - SYNTHETIC PROJECT-RISK DATASET")
print("=" * 60)

print("\nDataset shape:", project_data.shape)

print("\nTarget distributions:")
for target in [
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk"
]:
    print(f"\n{target}:")
    print(
        project_data[target]
        .value_counts()
        .sort_index()
        .to_string()
    )

print("\nMissing values:", project_data.isna().sum().sum())

print(
    "Duplicate Project IDs:",
    project_data["Project_ID"].duplicated().sum()
)

print(
    "Synthetic records:",
    int(project_data["Synthetic_Data_Flag"].sum()),
    "of",
    len(project_data)
)

print("\nSaved successfully to:")
print(output_path)

print("\nFirst five records:")
print(project_data.head().to_string(index=False))

print("\nDay 9 Step 3 completed.")


DAY 9 - SYNTHETIC PROJECT-RISK DATASET

Dataset shape: (300, 24)

Target distributions:

Project_Delayed:
Project_Delayed
0     38
1    262

Effort_Overrun:
Effort_Overrun
0    143
1    157

Low_Margin_Risk:
Low_Margin_Risk
0    221
1     79

Missing values: 0
Duplicate Project IDs: 0
Synthetic records: 300 of 300

Saved successfully to:
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\synthetic\Day9_Synthetic_Project_Risk_Dataset.csv

First five records:
Project_ID Project_Start_Date Target_End_Date Actual_End_Date  Planned_Effort  Actual_Effort  Milestones_Planned  Milestones_Completed  Change_Request_Count  Dependency_Delay_Days  Client_Delay_Days  Resource_Utilization  Project_Revenue  Forecast_Delivery_Cost  Actual_Delivery_Cost Data_Cutoff_Date  Delay_Days  Effort_Overrun_Pct  Forecast_Gross_Margin  Project_Delayed  Effort_Overrun  Low_Margin_Risk  Synthetic_Data_Flag                     Data_Use
  PRJ-0001         2023-01-01      2023-02-13      2023-03-01           854.0     

In [4]:

import pandas as pd
from pathlib import Path

# Load the synthetic dataset
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

data_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

df = pd.read_csv(data_path, parse_dates=[
    "Project_Start_Date",
    "Target_End_Date",
    "Actual_End_Date",
    "Data_Cutoff_Date"
])

# Define prediction targets
targets = [
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk"
]

# Exclude identifiers, post-outcome fields, target labels,
# and fields that are not valid as-of-cutoff predictors.
excluded_features = [
    "Project_ID",
    "Actual_End_Date",
    "Actual_Effort",
    "Actual_Delivery_Cost",
    "Delay_Days",
    "Effort_Overrun_Pct",
    "Forecast_Gross_Margin",
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk",
    "Synthetic_Data_Flag",
    "Data_Use",
    "Project_Revenue",
    "Forecast_Delivery_Cost",
    "Target_End_Date"
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_features
    and col not in targets
]

# Audit the remaining candidate features
feature_audit = pd.DataFrame({
    "Feature": feature_columns,
    "Data_Type": [
        str(df[col].dtype) for col in feature_columns
    ],
    "Missing_Values": [
        int(df[col].isna().sum()) for col in feature_columns
    ],
    "Unique_Values": [
        int(df[col].nunique()) for col in feature_columns
    ],
    "Availability_Check": [
        "Review required: confirm observed by Data_Cutoff_Date"
        for col in feature_columns
    ]
})

audit_path = (
    project_dir
    / "data"
    / "processed"
    / "Day9_Project_Risk_Feature_Audit.csv"
)

audit_path.parent.mkdir(parents=True, exist_ok=True)
feature_audit.to_csv(audit_path, index=False)

print("=" * 60)
print("DAY 9 - FEATURE LEAKAGE AUDIT")
print("=" * 60)

print("\nCandidate features:")
print(feature_audit.to_string(index=False))

print("\nTarget columns:")
print(targets)

print("\nFeature count:", len(feature_columns))

print("\nSaved feature audit to:")
print(audit_path)

print(
    "\nImportant: candidate features still require an "
    "as-of-cutoff availability review before modelling."
)


DAY 9 - FEATURE LEAKAGE AUDIT

Candidate features:
              Feature      Data_Type  Missing_Values  Unique_Values                                    Availability_Check
   Project_Start_Date datetime64[ns]               0            300 Review required: confirm observed by Data_Cutoff_Date
       Planned_Effort        float64               0            270 Review required: confirm observed by Data_Cutoff_Date
   Milestones_Planned          int64               0             17 Review required: confirm observed by Data_Cutoff_Date
 Milestones_Completed          int64               0             21 Review required: confirm observed by Data_Cutoff_Date
 Change_Request_Count          int64               0              9 Review required: confirm observed by Data_Cutoff_Date
Dependency_Delay_Days          int64               0             21 Review required: confirm observed by Data_Cutoff_Date
    Client_Delay_Days          int64               0             16 Review required: confirm ob

In [5]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

data_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

df = pd.read_csv(data_path, parse_dates=[
    "Project_Start_Date",
    "Target_End_Date",
    "Actual_End_Date",
    "Data_Cutoff_Date"
])

# Fixed seed so the correction is reproducible
rng = np.random.default_rng(43)

# Calculate planned project duration and elapsed time at cutoff
planned_duration = (
    df["Target_End_Date"] - df["Project_Start_Date"]
).dt.days.clip(lower=1)

elapsed_days = (
    df["Data_Cutoff_Date"] - df["Project_Start_Date"]
).dt.days.clip(lower=0)

progress = (
    elapsed_days / planned_duration
).clip(lower=0, upper=1).to_numpy()

# Milestones completed by the cutoff must not exceed planned milestones
milestones_planned = df["Milestones_Planned"].to_numpy()

milestones_completed = np.rint(
    milestones_planned * progress
    + rng.normal(0, 0.8, len(df))
).astype(int)

df["Milestones_Completed"] = np.clip(
    milestones_completed,
    0,
    milestones_planned
)

# Treat previously generated counts/delays as total project values.
# Estimate only the portion observed by the cutoff for this
# synthetic demonstration.
df["Change_Request_Count"] = rng.binomial(
    df["Change_Request_Count"].clip(lower=0).astype(int),
    np.clip(progress, 0, 1)
)

df["Dependency_Delay_Days"] = np.floor(
    df["Dependency_Delay_Days"] * progress
).astype(int)

df["Client_Delay_Days"] = np.floor(
    df["Client_Delay_Days"] * progress
).astype(int)

# Add elapsed project age as a cutoff-time feature
df["Project_Age_Days_At_Cutoff"] = elapsed_days

# Preserve the synthetic-data designation
df["Synthetic_Data_Flag"] = True
df["Data_Use"] = "Synthetic demonstration only"

# Save corrected dataset
df.to_csv(data_path, index=False)

# Rebuild preliminary feature audit
excluded_features = [
    "Project_ID",
    "Target_End_Date",
    "Actual_End_Date",
    "Actual_Effort",
    "Actual_Delivery_Cost",
    "Delay_Days",
    "Effort_Overrun_Pct",
    "Forecast_Gross_Margin",
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk",
    "Synthetic_Data_Flag",
    "Data_Use",
    "Project_Revenue",
    "Forecast_Delivery_Cost"
]

targets = [
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk"
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_features
    and col not in targets
]

feature_audit = pd.DataFrame({
    "Feature": feature_columns,
    "Data_Type": [
        str(df[col].dtype) for col in feature_columns
    ],
    "Missing_Values": [
        int(df[col].isna().sum()) for col in feature_columns
    ],
    "Unique_Values": [
        int(df[col].nunique()) for col in feature_columns
    ],
    "Review_Status": [
        "Synthetic cutoff-time feature; assumptions documented"
        for _ in feature_columns
    ]
})

audit_path = (
    project_dir
    / "data"
    / "processed"
    / "Day9_Project_Risk_Feature_Audit.csv"
)

audit_path.parent.mkdir(parents=True, exist_ok=True)
feature_audit.to_csv(audit_path, index=False)

# Validation
assert (
    df["Milestones_Completed"]
    <= df["Milestones_Planned"]
).all()

assert df["Milestones_Completed"].ge(0).all()
assert df["Change_Request_Count"].ge(0).all()
assert df["Dependency_Delay_Days"].ge(0).all()
assert df["Client_Delay_Days"].ge(0).all()

print("DAY 9 - CUTOFF FEATURE CHECK")
print("Dataset shape:", df.shape)
print(
    "Milestones completed <= milestones planned:",
    bool(
        (
            df["Milestones_Completed"]
            <= df["Milestones_Planned"]
        ).all()
    )
)
print("Missing values:", int(df.isna().sum().sum()))
print("Feature count:", len(feature_columns))
print("\nFeature audit:")
print(feature_audit.to_string(index=False))
print("\nUpdated dataset:", data_path)
print("Updated audit:", audit_path)
print("\nSynthetic demonstration only; real-world validation pending.")


DAY 9 - CUTOFF FEATURE CHECK
Dataset shape: (300, 25)
Milestones completed <= milestones planned: True
Missing values: 0
Feature count: 10

Feature audit:
                   Feature      Data_Type  Missing_Values  Unique_Values                                         Review_Status
        Project_Start_Date datetime64[ns]               0            300 Synthetic cutoff-time feature; assumptions documented
            Planned_Effort        float64               0            270 Synthetic cutoff-time feature; assumptions documented
        Milestones_Planned          int64               0             17 Synthetic cutoff-time feature; assumptions documented
      Milestones_Completed          int64               0             13 Synthetic cutoff-time feature; assumptions documented
      Change_Request_Count          int64               0              7 Synthetic cutoff-time feature; assumptions documented
     Dependency_Delay_Days          int64               0             13 Synthetic 

In [6]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Load the corrected synthetic dataset
# ------------------------------------------------------------
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

data_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

df = pd.read_csv(
    data_path,
    parse_dates=["Project_Start_Date", "Data_Cutoff_Date"]
)

# ------------------------------------------------------------
# 2. Define cutoff-time predictors and targets
# ------------------------------------------------------------
feature_columns = [
    "Planned_Effort",
    "Milestones_Planned",
    "Milestones_Completed",
    "Change_Request_Count",
    "Dependency_Delay_Days",
    "Client_Delay_Days",
    "Resource_Utilization",
    "Project_Age_Days_At_Cutoff"
]

targets = [
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk"
]

# ------------------------------------------------------------
# 3. Time-aware split: earliest 80% train, latest 20% test
# ------------------------------------------------------------
df = df.sort_values(
    ["Project_Start_Date", "Project_ID"]
).reset_index(drop=True)

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

print("TRAIN/TEST SPLIT")
print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print(
    "Training start-date range:",
    train_df["Project_Start_Date"].min(),
    "to",
    train_df["Project_Start_Date"].max()
)
print(
    "Testing start-date range:",
    test_df["Project_Start_Date"].min(),
    "to",
    test_df["Project_Start_Date"].max()
)

# ------------------------------------------------------------
# 4. Define baseline and stronger candidate models
# ------------------------------------------------------------
baseline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

candidate = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ))
])

# ------------------------------------------------------------
# 5. Evaluate each target separately
# ------------------------------------------------------------
results = []
confusion_rows = []

for target in targets:
    y_train = train_df[target].astype(int)
    y_test = test_df[target].astype(int)

    print("\n" + "=" * 60)
    print("TARGET:", target)
    print("Training class counts:")
    print(y_train.value_counts().sort_index().to_dict())
    print("Testing class counts:")
    print(y_test.value_counts().sort_index().to_dict())

    if y_train.nunique() < 2 or y_test.nunique() < 2:
        print(
            "SKIPPED: both classes must be present in "
            "training and testing data for this evaluation."
        )
        continue

    for model_name, model in [
        ("Logistic Regression", baseline),
        ("Random Forest", candidate)
    ]:
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1]

        metrics = {
            "Target": target,
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC_AUC": roc_auc_score(y_test, probabilities),
            "PR_AUC": average_precision_score(
                y_test, probabilities
            ),
            "Train_Rows": len(train_df),
            "Test_Rows": len(test_df),
            "Data_Status": "Synthetic demonstration only"
        }

        results.append(metrics)

        cm = confusion_matrix(
            y_test, predictions, labels=[0, 1]
        )

        confusion_rows.append({
            "Target": target,
            "Model": model_name,
            "TN": int(cm[0, 0]),
            "FP": int(cm[0, 1]),
            "FN": int(cm[1, 0]),
            "TP": int(cm[1, 1])
        })

        print("\nModel:", model_name)
        print("Accuracy:", round(metrics["Accuracy"], 4))
        print("Precision:", round(metrics["Precision"], 4))
        print("Recall:", round(metrics["Recall"], 4))
        print("F1:", round(metrics["F1"], 4))
        print("ROC-AUC:", round(metrics["ROC_AUC"], 4))
        print("PR-AUC:", round(metrics["PR_AUC"], 4))
        print("Confusion matrix [[TN, FP], [FN, TP]]:")
        print(cm)

# ------------------------------------------------------------
# 6. Save evaluation results
# ------------------------------------------------------------
processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

if results:
    results_df = pd.DataFrame(results)
    results_path = (
        processed_dir / "Day9_Project_Risk_Model_Comparison.csv"
    )
    results_df.to_csv(results_path, index=False)

    print("\nMODEL COMPARISON")
    print(results_df.round(4).to_string(index=False))
    print("\nSaved:", results_path)

if confusion_rows:
    confusion_df = pd.DataFrame(confusion_rows)
    confusion_path = (
        processed_dir / "Day9_Project_Risk_Confusion_Matrices.csv"
    )
    confusion_df.to_csv(confusion_path, index=False)

    print("\nSaved confusion matrices:", confusion_path)

print(
    "\nReminder: results measure performance on synthetic data, "
    "not real NexziByte project outcomes."
)


TRAIN/TEST SPLIT
Training records: 240
Testing records: 60
Training start-date range: 2023-01-01 00:00:00 to 2025-05-01 00:00:00
Testing start-date range: 2025-05-04 00:00:00 to 2025-12-01 00:00:00

TARGET: Project_Delayed
Training class counts:
{0: 33, 1: 207}
Testing class counts:
{0: 5, 1: 55}

Model: Logistic Regression
Accuracy: 0.7833
Precision: 1.0
Recall: 0.7636
F1: 0.866
ROC-AUC: 0.9418
PR-AUC: 0.9949
Confusion matrix [[TN, FP], [FN, TP]]:
[[ 5  0]
 [13 42]]

Model: Random Forest
Accuracy: 0.8167
Precision: 0.9783
Recall: 0.8182
F1: 0.8911
ROC-AUC: 0.8836
PR-AUC: 0.9893
Confusion matrix [[TN, FP], [FN, TP]]:
[[ 4  1]
 [10 45]]

TARGET: Effort_Overrun
Training class counts:
{0: 110, 1: 130}
Testing class counts:
{0: 33, 1: 27}

Model: Logistic Regression
Accuracy: 0.8
Precision: 0.7586
Recall: 0.8148
F1: 0.7857
ROC-AUC: 0.8653
PR-AUC: 0.8604
Confusion matrix [[TN, FP], [FN, TP]]:
[[26  7]
 [ 5 22]]

Model: Random Forest
Accuracy: 0.7667
Precision: 0.7241
Recall: 0.7778
F1: 0.75

In [8]:

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# DAY 9 - STEP 6: PROJECT RISK REGISTER AND REPORTS
# ============================================================

# 1. Load dataset and define directories
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

data_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

if not data_path.exists():
    raise FileNotFoundError(
        f"Required synthetic dataset not found: {data_path}"
    )

df = pd.read_csv(
    data_path,
    parse_dates=["Project_Start_Date", "Data_Cutoff_Date"]
)

processed_dir = project_dir / "data" / "processed"
reports_dir = project_dir / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# 2. Validate required columns
required_columns = [
    "Project_ID",
    "Project_Start_Date",
    "Data_Cutoff_Date",
    "Planned_Effort",
    "Milestones_Planned",
    "Milestones_Completed",
    "Change_Request_Count",
    "Dependency_Delay_Days",
    "Client_Delay_Days",
    "Resource_Utilization",
    "Project_Age_Days_At_Cutoff",
    "Project_Delayed",
    "Effort_Overrun",
    "Forecast_Gross_Margin",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {missing_columns}"
    )

if df.empty:
    raise ValueError("The project risk dataset is empty.")

if df["Project_ID"].isna().any():
    raise ValueError("Project_ID contains missing values.")

if df["Project_ID"].duplicated().any():
    raise ValueError(
        "Duplicate Project_ID values found in the source dataset."
    )

if df["Project_Delayed"].isna().any():
    raise ValueError("Project_Delayed contains missing target values.")

if df["Effort_Overrun"].isna().any():
    raise ValueError("Effort_Overrun contains missing target values.")

if df["Forecast_Gross_Margin"].isna().any():
    raise ValueError(
        "Forecast_Gross_Margin contains missing values."
    )

# 3. Sort chronologically and split data
df = df.sort_values(
    ["Project_Start_Date", "Project_ID"]
).reset_index(drop=True)

if len(df) < 5:
    raise ValueError(
        "At least five records are required for this demonstration."
    )

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

if train_df.empty or test_df.empty:
    raise ValueError(
        "Chronological train/test split produced an empty partition."
    )

features = [
    "Planned_Effort",
    "Milestones_Planned",
    "Milestones_Completed",
    "Change_Request_Count",
    "Dependency_Delay_Days",
    "Client_Delay_Days",
    "Resource_Utilization",
    "Project_Age_Days_At_Cutoff",
]

X_train = train_df[features]
X_test = test_df[features]

# 4. Train selected delay and effort models
delay_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    )),
])

effort_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )),
])

# Ensure both target classes exist in each training target.
for target_column in ["Project_Delayed", "Effort_Overrun"]:
    if train_df[target_column].nunique() < 2:
        raise ValueError(
            f"{target_column} has fewer than two classes in the "
            "training partition. Model training cannot proceed."
        )

delay_model.fit(
    X_train,
    train_df["Project_Delayed"].astype(int),
)

effort_model.fit(
    X_train,
    train_df["Effort_Overrun"].astype(int),
)

delay_probability = delay_model.predict_proba(X_test)[:, 1]
effort_probability = effort_model.predict_proba(X_test)[:, 1]

# 5. Create risk register using the complete test rows
# Preserve original test rows and the fields needed for reason codes.
risk_register = test_df.copy().reset_index(drop=True)

# Store probabilities rounded to four decimal places.
risk_register["Delay_Risk_Probability"] = np.round(
    delay_probability,
    4,
)

risk_register["Effort_Overrun_Probability"] = np.round(
    effort_probability,
    4,
)

# Risk bands:
# High   = probability >= 0.70
# Medium = probability >= 0.40 and < 0.70
# Low    = probability < 0.40
#
# These are provisional demonstration bands, not business-approved
# operational thresholds.

def risk_level(probability):
    if probability >= 0.70:
        return "High"
    elif probability >= 0.40:
        return "Medium"
    return "Low"


# IMPORTANT:
# Derive risk labels from the rounded probabilities stored in the
# register so displayed probabilities and labels remain consistent.
risk_register["Delay_Risk_Level"] = (
    risk_register["Delay_Risk_Probability"].apply(risk_level)
)

risk_register["Effort_Risk_Level"] = (
    risk_register["Effort_Overrun_Probability"].apply(risk_level)
)

# 6. Rule-based margin assessment
# The 20% threshold is illustrative and requires finance approval.
margin_threshold = 0.20

risk_register["Forecast_Gross_Margin_Pct"] = (
    risk_register["Forecast_Gross_Margin"] * 100
).round(2)

risk_register["Margin_Risk_Level"] = np.where(
    risk_register["Forecast_Gross_Margin"] < margin_threshold,
    "High",
    "Lower",
)

risk_register["Margin_Assessment_Method"] = np.where(
    risk_register["Forecast_Gross_Margin"] < margin_threshold,
    "Rule-based: forecast gross margin below illustrative 20% threshold",
    "Rule-based: forecast gross margin at or above illustrative 20% threshold",
)

# 7. Create transparent, rule-based reason codes
def create_reason_codes(row):
    reasons = []

    planned = max(int(row["Milestones_Planned"]), 1)
    completed = int(row["Milestones_Completed"])

    if completed / planned < 0.50:
        reasons.append("LOW_MILESTONE_PROGRESS")

    if row["Change_Request_Count"] >= 3:
        reasons.append("HIGH_CHANGE_REQUESTS")

    if row["Dependency_Delay_Days"] >= 7:
        reasons.append("DEPENDENCY_DELAY")

    if row["Client_Delay_Days"] >= 5:
        reasons.append("CLIENT_DELAY")

    if row["Resource_Utilization"] >= 0.90:
        reasons.append("HIGH_RESOURCE_UTILIZATION")

    return (
        "; ".join(reasons)
        if reasons
        else "NO_RULE_TRIGGERED"
    )


risk_register["Reason_Codes"] = risk_register.apply(
    create_reason_codes,
    axis=1,
)

risk_register["Data_Status"] = (
    "Synthetic demonstration only; not operational"
)

# 8. Save the project risk register
register_path = (
    processed_dir / "Day9_Project_Risk_Register.csv"
)

risk_register.to_csv(
    register_path,
    index=False,
)

# 9. Generate the margin-risk report
margin_high = int(
    (risk_register["Margin_Risk_Level"] == "High").sum()
)

margin_lower = int(
    (risk_register["Margin_Risk_Level"] == "Lower").sum()
)

margin_report = f"""# Day 9 - Margin Risk Report

## Status
Synthetic demonstration only; real-business validation pending.

## Method
Forecast gross margin = (Project Revenue - Forecast Delivery Cost)
/ Project Revenue.

Projects with forecast gross margin below 20% are flagged as high
margin risk. The 20% threshold is illustrative and pending finance
approval; it is not an approved NexziByte business rule.

## Test-set summary
- Test projects: {len(risk_register)}
- Below the illustrative 20% margin threshold: {margin_high}
- At or above the illustrative 20% threshold: {margin_lower}

## Model evaluation
The low-margin classifier performed poorly on the synthetic
time-separated test set. The previously recorded evaluation showed:
- Logistic Regression ROC-AUC: 0.3681
- Random Forest ROC-AUC: 0.4045

Neither model is recommended as a reliable margin-risk predictor.
These metrics describe the earlier synthetic evaluation and must
not be interpreted as real-business performance.

## Limitations
- Revenue and delivery costs are synthetic.
- The margin threshold requires finance approval.
- Margin risk in this register is rule-based, not a calibrated
  model probability.
- Actual delivery cost is an outcome field, not a predictor.
- No real project-level financial records were found in the
  audited datasets.

## Required validation
Obtain approved project revenue, forecast delivery costs, the
official gross-margin definition, and a finance-approved threshold.
Validate the margin calculations against real, dated project data
before considering operational use.
"""

margin_report_path = (
    reports_dir / "Day9_Margin_Risk_Report.md"
)

margin_report_path.write_text(
    margin_report,
    encoding="utf-8",
)

# 10. Generate the project-risk summary report
high_delay = int(
    (risk_register["Delay_Risk_Level"] == "High").sum()
)

medium_delay = int(
    (risk_register["Delay_Risk_Level"] == "Medium").sum()
)

low_delay = int(
    (risk_register["Delay_Risk_Level"] == "Low").sum()
)

high_effort = int(
    (risk_register["Effort_Risk_Level"] == "High").sum()
)

medium_effort = int(
    (risk_register["Effort_Risk_Level"] == "Medium").sum()
)

low_effort = int(
    (risk_register["Effort_Risk_Level"] == "Low").sum()
)

summary_report = f"""# Day 9 - Project Risk Summary

## Status
Synthetic demonstration complete; business validation pending.

## Evaluation design
- Total synthetic records: {len(df)}
- Training records: {len(train_df)}
- Test records: {len(test_df)}
- Split: chronological by project start date

## Test-set delay-risk register
- High delay-risk flags: {high_delay}
- Medium delay-risk flags: {medium_delay}
- Low delay-risk flags: {low_delay}

## Test-set effort-overrun-risk register
- High effort-overrun-risk flags: {high_effort}
- Medium effort-overrun-risk flags: {medium_effort}
- Low effort-overrun-risk flags: {low_effort}

## Test-set margin assessment
- Projects below the illustrative 20% forecast gross-margin threshold: {margin_high}
- Projects at or above the illustrative 20% threshold: {margin_lower}

## Model selection
- Delay: Random Forest, based on the previously recorded higher
  synthetic test-set F1 score.
- Effort overrun: Logistic Regression, based on the previously
  recorded higher synthetic test-set F1 score.
- Low-margin classifier: not recommended due to previously recorded
  ROC-AUC below 0.50.

The delay and effort models were retrained by this code. The model
selection statements above refer to the earlier evaluation results;
rerun model evaluation if the dataset or split has changed.

## Risk classification
- High: probability >= 0.70
- Medium: probability >= 0.40 and < 0.70
- Low: probability < 0.40

These are provisional demonstration thresholds, not approved
operational business rules.

## Reason codes
Reason codes are transparent, rule-based indicators. They are not
SHAP explanations, causal findings, or proof of the cause of an outcome.

## Limitations and next steps
All records and outcomes are synthetic. Test metrics do not establish
real-world predictive performance. Verify cutoff-time availability,
obtain stakeholder approval for target definitions, and validate
against real dated project data before operational use.
"""

summary_path = (
    reports_dir / "Day9_Project_Risk_Summary.md"
)

summary_path.write_text(
    summary_report,
    encoding="utf-8",
)

# 11. Validate the risk register and saved outputs
print("=" * 65)
print("DAY 9 - PROJECT RISK OUTPUTS")
print("=" * 65)

print("\nDataset summary")
print("Total source records:", len(df))
print("Training records:", len(train_df))
print("Test records:", len(test_df))

print("\nRisk register validation")
print("Risk register shape:", risk_register.shape)
print(
    "Missing values:",
    int(risk_register.isna().sum().sum()),
)
print(
    "Duplicate Project IDs:",
    int(risk_register["Project_ID"].duplicated().sum()),
)

print("\nDelay risk levels")
print(
    risk_register["Delay_Risk_Level"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .to_string()
)

print("\nEffort-overrun risk levels")
print(
    risk_register["Effort_Risk_Level"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .to_string()
)

print("\nMargin risk levels")
print(
    risk_register["Margin_Risk_Level"]
    .value_counts()
    .reindex(["High", "Lower"], fill_value=0)
    .to_string()
)

print("\nRisk-level consistency checks")

expected_delay_levels = risk_register[
    "Delay_Risk_Probability"
].apply(risk_level)

expected_effort_levels = risk_register[
    "Effort_Overrun_Probability"
].apply(risk_level)

delay_mismatches = int(
    (
        risk_register["Delay_Risk_Level"]
        != expected_delay_levels
    ).sum()
)

effort_mismatches = int(
    (
        risk_register["Effort_Risk_Level"]
        != expected_effort_levels
    ).sum()
)

print("Delay risk-level mismatches:", delay_mismatches)
print("Effort risk-level mismatches:", effort_mismatches)

print("\nMargin-threshold consistency check")

expected_margin_levels = np.where(
    risk_register["Forecast_Gross_Margin"] < margin_threshold,
    "High",
    "Lower",
)

margin_mismatches = int(
    (
        risk_register["Margin_Risk_Level"]
        != expected_margin_levels
    ).sum()
)

print("Margin risk-level mismatches:", margin_mismatches)

print("\nReason-code examples")
print(
    risk_register[
        ["Project_ID", "Reason_Codes"]
    ].head(10).to_string(index=False)
)

print("\nSaved files")

for path in [
    register_path,
    margin_report_path,
    summary_path,
]:
    print(
        f"{path}: "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

# Final integrity assertions
assert risk_register["Project_ID"].is_unique, (
    "Risk register contains duplicate Project IDs."
)

assert delay_mismatches == 0, (
    "Delay risk-level consistency check failed."
)

assert effort_mismatches == 0, (
    "Effort risk-level consistency check failed."
)

assert margin_mismatches == 0, (
    "Margin risk-level consistency check failed."
)

assert register_path.exists(), "Risk register CSV was not saved."
assert margin_report_path.exists(), "Margin report was not saved."
assert summary_path.exists(), "Project-risk summary was not saved."

print("\nAll risk-level consistency checks passed.")
print("Synthetic demonstration only; not approved for operational use.")


DAY 9 - PROJECT RISK OUTPUTS

Dataset summary
Total source records: 300
Training records: 240
Test records: 60

Risk register validation
Risk register shape: (60, 34)
Missing values: 0
Duplicate Project IDs: 0

Delay risk levels
Delay_Risk_Level
High      26
Medium    25
Low        9

Effort-overrun risk levels
Effort_Risk_Level
High      16
Medium    17
Low       27

Margin risk levels
Margin_Risk_Level
High     12
Lower    48

Risk-level consistency checks
Delay risk-level mismatches: 0
Effort risk-level mismatches: 0

Margin-threshold consistency check
Margin risk-level mismatches: 0

Reason-code examples
Project_ID                                Reason_Codes
  PRJ-0241                            DEPENDENCY_DELAY
  PRJ-0242 DEPENDENCY_DELAY; HIGH_RESOURCE_UTILIZATION
  PRJ-0243              DEPENDENCY_DELAY; CLIENT_DELAY
  PRJ-0244                           NO_RULE_TRIGGERED
  PRJ-0245              DEPENDENCY_DELAY; CLIENT_DELAY
  PRJ-0246                                CLIENT_DELAY